# FactorCalculator 使用示例

本 Notebook 演示 `factor_calculator` 包的各种用法，包括解析单元规格、创建实例、以及使用计算器进行因子计算。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

## 1. 解析单元规格（Unit Specifications）

In [ ]:
from factor_calculator import parse_unit_spec

specs = [
    "MdDMU",
    "KlineDMU(15)",
    # "KlineDMU(interval=5)",
    # "BiquotePEU(watching_time=60)",
    # "PositionPnlDMU",
]

for spec in specs:
    class_name, params = parse_unit_spec(spec)
    print(f"{spec!r}")
    print(f"  -> class: {class_name}, params: {params!r}")

## 2. 列出可用的 DMU / PEU 类

> 需要安装 `rbt` 包才能运行此部分。

In [ ]:
from factor_calculator import get_available_classes

print("所有可用单元:")
for cls in get_available_classes():
    print(f"  - {cls}")

print("\nDMU 类:")
for cls in get_available_classes("DMU"):
    print(f"  - {cls}")

print("\nPEU 类:")
for cls in get_available_classes("PEU"):
    print(f"  - {cls}")

## 3. 创建单元实例

> 需要安装 `rbt` 包才能运行此部分。

In [ ]:
from factor_calculator import create_unit, create_units

# 创建单个单元
unit = create_unit("KlineDMU(interval=5)")
print(f"Created: {unit}")
print(f"  Name: {unit.name}")
print(f"  Interval: {unit.interval}")

In [ ]:
# 批量创建多个单元
units = create_units(["PositionPnlDMU", "FixedHoldingPEU(watching_mds=100)", "TrendDMU"])
print(f"创建了 {len(units)} 个单元:")
for u in units:
    print(f"  - {u.name}")

## 4. FactorCalculator（完整集成）

完整版计算器，集成 RBT Strategy 引擎、行情加载、结果数据库。

In [ ]:
from factor_calculator import FactorCalculator
import datetime

#初始化（root_path 替代了旧的 db_directory，新增 frequency 参数）
calc = FactorCalculator()

#执行因子计算
result = calc.calculate(
    units=[
        "MdDMU()",
        "KlineDMU(interval=15)",
        # "BiquotePEU(1)",
        # "FixedHoldingPEU(10)",
    ],
    load_factors=[],
    contract="T2603",
    trade_date=datetime.date(2026, 1, 6),
    frequency="tick",
)
result

## 4.1 读取 Parquet 数据

In [6]:
import pandas as pd

parquet_path = "~/data/factor_store/tick/T2603/2026-01-06/MdDMU_v0.parquet"  # <- 改成你的 parquet 路径

df = pd.read_parquet(parquet_path)
df

,ts,MdDMU_v0__bid,MdDMU_v0__ask,MdDMU_v0__mid,MdDMU_v0__mid_smo,MdDMU_v0__mean,MdDMU_v0__std,MdDMU_v0__quantile,MdDMU_v0__ob_avg,MdDMU_v0__cum_avg,MdDMU_v0__exec_avg
0,2026-01-06 09:29:00.300,107.700,107.705,107.7025,107.7025,107.70250,0.000000,0.000000,107.704661,107.700000,NaN
1,2026-01-06 09:30:00.300,107.710,107.740,107.7250,107.7250,107.71375,0.011250,1.000000,107.725000,107.702654,107.709526
2,2026-01-06 09:30:00.800,107.720,107.730,107.7250,107.7250,107.71750,0.010607,0.707107,107.721667,107.707894,107.719063
3,2026-01-06 09:30:01.300,107.715,107.730,107.7225,107.7225,107.71875,0.009437,0.397360,107.719333,107.710526,107.722850
4,2026-01-06 09:30:01.800,107.715,107.720,107.7175,107.7225,107.71950,0.008573,0.349927,107.718750,107.711517,107.716798
...,...,...,...,...,...,...,...,...,...,...,...
23666,2026-01-06 15:14:58.300,107.695,107.700,107.6975,107.6975,107.69750,0.000000,0.000000,107.696266,107.696485,107.700000
23667,2026-01-06 15:14:59.300,107.695,107.700,107.6975,107.6975,107.69750,0.000000,0.000000,107.696282,107.696485,107.700000
23668,2026-01-06 15:14:59.800,107.695,107.700,107.6975,107.6975,107.69750,0.000000,0.000000,107.696234,107.696485,NaN
23669,2026-01-06 15:15:00.300,107.695,107.700,107.6975,107.6975,107.69750,0.000000,0.000000,107.695862,107.696486,107.700000


## 5. CLI 命令行用法

安装包后可直接在终端使用 `factor-calculator` 命令：

```bash
# 列出可用单元
factor-calculator list
factor-calculator list --dmu
factor-calculator list --peu

# 计算因子
factor-calculator calculate \
    --db /path/to/results \
    --md /path/to/market/data \
    --units "KlineDMU(5),BiquotePEU(60)" \
    --contract IF2403 \
    --date 2024-03-15

# 查看已有因子
factor-calculator factors \
    --db /path/to/results \
    --contract IF2403 \
    --date 2024-03-15
```